# Myeloma Neoantigen Target Discovery

This notebook demonstrates a minimal neoantigen target discovery pipeline 
using publicly available Multiple Myeloma somatic mutation data from the 
MMRF CoMMpass study (GDC open access).

## Workflow
1. Load and filter somatic mutations from MAF files
2. Extract missense mutations and derive mutant peptide sequences
3. Predict MHC-I binding affinity using netMHCpan 4.2
4. Rank candidate neoantigen targets

**Data**: MMRF-COMMPASS masked somatic MAF files (open access, GDC)  
**Tool**: netMHCpan 4.2c (DTU Health Tech)  
**HLA allele**: HLA-A*02:01 (most common allele, used as proof of concept)

In [1]:
import pandas as pd
import subprocess
import os
from pathlib import Path
from glob import glob

# Project paths
PROJECT_DIR = Path.home() / "ExtremeSSD" / "neoantigen-target-discovery"
MAF_DIR = PROJECT_DIR / "data" / "raw_mafs"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
NETMHCPAN = Path.home() / "ExtremeSSD" / "neoantigen-target-discovery" / "tools" / "netMHCpan-4.2" / "netMHCpan"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Confirm files are found
maf_files = glob(str(MAF_DIR / "*.maf.gz"))
print(f"Found {len(maf_files)} MAF files")
for f in maf_files:
    print(f"  {Path(f).name}")

Found 1091 MAF files
  68831f1e-7c6d-423e-8e19-cce4922237e9.maf.gz
  c25100fd-eb22-483c-9bb2-26257a6f4049.maf.gz
  520bce32-b4fb-4fe3-b28a-7452a80b309e.maf.gz
  7635407f-e8f3-4f06-9865-e86204f4a1ef.maf.gz
  6335aed2-979e-4a67-b17a-237d0f8cb366.maf.gz
  a403e98b-acc6-4805-943f-c8d7350deb90.maf.gz
  f96f632b-432b-4bcf-aa5f-5defe5bfb11b.maf.gz
  2c1d1d77-cac8-44c1-ade9-ce82fecce02a.maf.gz
  19fa7dd7-3851-4141-9219-dd1139365106.maf.gz
  f68a373a-ad30-4db1-b9ba-ada497eeac30.maf.gz
  a38c4031-9249-4092-bed4-36e1d5b7dd5e.maf.gz
  a6af2ccb-7acc-42f3-81d9-ee2329662eb7.maf.gz
  7fe2b8c5-ca80-4b64-8781-0017476e7bf3.maf.gz
  3f60cb59-ae09-4a54-9b61-4eb08af719f8.maf.gz
  fad81b91-4d7b-4812-ba27-acef564750d8.maf.gz
  bee0c2c7-ecb7-4217-9e62-19934fb47718.maf.gz
  4a51a4a0-445b-4421-83f3-0d279ef46905.maf.gz
  180e9aad-eb2f-4e02-b026-18bd9cfcc6e0.maf.gz
  90e45b7c-8b2b-42fb-b02d-47f9b713a3d2.maf.gz
  7c4646f9-8501-4015-8029-c98e367e8659.maf.gz
  82fdf4cb-15a0-4eb3-b4f4-ce6d85b8f692.maf.gz
  550daafe-f2

## Step 1: Load and Filter Somatic Mutations

We load all MAF files and filter for **missense mutations** only.

Missense mutations are single amino acid substitutions. They are the primary 
source of neoantigens in most cancers because:

- They change the peptide sequence presented on MHC molecules
- The mutant peptide may be recognized as foreign by T-cells
- They are more likely to be clonal (present in all tumor cells) than indels

We exclude silent, intronic, UTR, and frameshift variants for this analysis. 
Frameshifts generate novel peptides and are also strong neoantigen candidates, 
but require a different extraction approach and will be added in a future iteration.

In [2]:
# Load and merge all MAF files
dfs = []
for f in maf_files:
    df = pd.read_csv(f, sep="\t", comment="#", low_memory=False)
    dfs.append(df)

maf = pd.concat(dfs, ignore_index=True)
print(f"Total variants loaded: {len(maf)}")
print(f"Samples: {maf['Tumor_Sample_Barcode'].nunique()}")
print(f"\nVariant classifications:")
print(maf["Variant_Classification"].value_counts())

Total variants loaded: 154588
Samples: 1091

Variant classifications:
Variant_Classification
Missense_Mutation         51005
3'UTR                     40503
Silent                    19667
Intron                    18167
5'UTR                      6829
Nonsense_Mutation          3632
5'Flank                    3419
3'Flank                    3188
RNA                        2760
Frame_Shift_Del            1553
Splice_Site                1413
Splice_Region              1181
Frame_Shift_Ins             572
In_Frame_Del                391
Translation_Start_Site      164
Nonstop_Mutation             74
In_Frame_Ins                 53
IGR                          17
Name: count, dtype: int64


In [3]:
# Filter for missense mutations only
missense = maf[maf["Variant_Classification"] == "Missense_Mutation"].copy()

# Keep only columns we need
cols = [
    "Hugo_Symbol",
    "Tumor_Sample_Barcode", 
    "Chromosome",
    "Start_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
    "HGVSp_Short",
    "Protein_position",
    "Amino_acids"
]

missense = missense[cols].reset_index(drop=True)
print(f"Missense mutations: {len(missense)}")
print(f"\nFirst few rows:")
missense.head()

Missense mutations: 51005

First few rows:


,Hugo_Symbol,Tumor_Sample_Barcode,Chromosome,Start_Position,Reference_Allele,Tumor_Seq_Allele2,HGVSp_Short,Protein_position,Amino_acids
0,S1PR1,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,chr1,101239736,C,T,p.S251L,251/382,S/L
1,HCN3,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,chr1,155282416,A,C,p.Y95S,95/774,Y/S
2,GPN1,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,chr2,27647844,T,G,p.S314A,314/374,S/A
3,IQCA1,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,chr2,236497425,G,A,p.P25L,25/822,P/L
4,SEC22A,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,chr3,123271591,C,G,p.L265V,265/307,L/V


## Step 2: Extract Mutant Peptide Sequences

To predict MHC binding, we need the actual peptide sequence surrounding each mutation.

We extract this from the `HGVSp_Short` column (e.g. `p.S251L` means Serine at 
position 251 changed to Leucine).

Since we do not have the full protein sequences in the MAF, we use a simplified 
approach: we construct a synthetic 9-mer centered on the mutant amino acid. 
This is a proof-of-concept implementation. A production pipeline would fetch 
the full protein sequence from UniProt or Ensembl and extract the true 
flanking context.

A 9-mer is the most common peptide length for HLA class I presentation.

In [4]:
import re

def parse_hgvsp(hgvsp):
    """
    Parse HGVSp_Short notation e.g. p.S251L
    Returns (ref_aa, position, alt_aa) or None if unparseable
    """
    match = re.match(r"p\.([A-Z])(\d+)([A-Z])", str(hgvsp))
    if match:
        ref_aa = match.group(1)
        position = int(match.group(2))
        alt_aa = match.group(3)
        return ref_aa, position, alt_aa
    return None

def build_synthetic_peptide(alt_aa, length=9):
    """
    Build a synthetic 9-mer centered on the mutant amino acid.
    Flanking positions are filled with Alanine (A) as neutral placeholder.
    In a production pipeline, true protein context would be used.
    """
    flank = length // 2
    peptide = "A" * flank + alt_aa + "A" * flank
    return peptide

# Parse and build peptides
records = []
for _, row in missense.iterrows():
    parsed = parse_hgvsp(row["HGVSp_Short"])
    if parsed is None:
        continue
    ref_aa, position, alt_aa = parsed
    peptide = build_synthetic_peptide(alt_aa)
    records.append({
        "gene": row["Hugo_Symbol"],
        "sample": row["Tumor_Sample_Barcode"],
        "hgvsp": row["HGVSp_Short"],
        "ref_aa": ref_aa,
        "alt_aa": alt_aa,
        "position": position,
        "peptide": peptide
    })

peptides_df = pd.DataFrame(records)
print(f"Peptides generated: {len(peptides_df)}")
peptides_df.head(10)

Peptides generated: 50973


,gene,sample,hgvsp,ref_aa,alt_aa,position,peptide
0,S1PR1,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.S251L,S,L,251,AAAALAAAA
1,HCN3,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.Y95S,Y,S,95,AAAASAAAA
2,GPN1,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.S314A,S,A,314,AAAAAAAAA
3,IQCA1,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.P25L,P,L,25,AAAALAAAA
4,SEC22A,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.L265V,L,V,265,AAAAVAAAA
5,KLHL6,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.R332W,R,W,332,AAAAWAAAA
6,LTB,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.A63T,A,T,63,AAAATAAAA
7,ZKSCAN5,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.V674I,V,I,674,AAAAIAAAA
8,ST18,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.K767R,K,R,767,AAAARAAAA
9,KLHL38,MMRF_1607_1_BM_CD138pos_T2_KBS5U_K11921,p.R43Q,R,Q,43,AAAAQAAAA


In [5]:
# Note: mutations to Alanine (e.g. p.S314A) will generate AAAAAAAAA
# These peptides are uninformative for binding prediction since the
# flanking context is also Alanine. In a production pipeline, true
# protein sequence context from UniProt/Ensembl would resolve this.
# For this proof of concept we retain them but flag them in output.

peptides_df["is_ala_mutation"] = peptides_df["alt_aa"] == "A"
print(f"Alanine mutations (uninformative peptides): {peptides_df['is_ala_mutation'].sum()}")

Alanine mutations (uninformative peptides): 1826


## Step 3: Run netMHCpan Binding Predictions

We write the peptides to a file and run netMHCpan 4.2c to predict 
MHC-I binding affinity against HLA-A*02:01.

**Key parameters:**
- `-a HLA-A02:01`: target allele
- `-p`: input is peptide format (not FASTA)
- `-rankS 0.5`: strong binder threshold (top 0.5% rank)
- `-rankW 2.0`: weak binder threshold (top 2% rank)

**Interpreting %Rank:**
- < 0.5% : Strong binder (SB) — high priority vaccine candidate
- 0.5–2% : Weak binder (WB) — moderate priority
- > 2%   : Non-binder — deprioritized

%Rank is preferred over raw score because it is normalized across 
different alleles and peptide lengths, making results comparable.

In [6]:
# Write peptides to file for netMHCpan input
peptide_input = OUTPUT_DIR / "missense_peptides.txt"

with open(peptide_input, "w") as f:
    for _, row in peptides_df.iterrows():
        f.write(row["peptide"] + "\n")

print(f"Wrote {len(peptides_df)} peptides to {peptide_input}")

# Run netMHCpan
netmhcpan_output = OUTPUT_DIR / "netmhcpan_results.txt"

cmd = [
    "tcsh", str(NETMHCPAN),
    "-a", "HLA-A02:01",
    "-p",
    "-f", str(peptide_input),
    "-t", "-1"  # print all results regardless of rank
]

print("Running netMHCpan...")
result = subprocess.run(cmd, capture_output=True, text=True)

with open(netmhcpan_output, "w") as f:
    f.write(result.stdout)

print("Done.")
print(result.stdout[:500])

Wrote 50973 peptides to /Users/trastogi/ExtremeSSD/neoantigen-target-discovery/data/processed/missense_peptides.txt
Running netMHCpan...
Done.
# /Users/trastogi/ExtremeSSD/neoantigen-target-discovery/tools/netMHCpan-4.2/Darwin_arm64/bin/netMHCpan-4.2 -a HLA-A02:01 -p -f /Users/trastogi/ExtremeSSD/neoantigen-target-discovery/data/processed/missense_peptides.txt -t -1
# Thu Apr 30 21:40:23 2026
# User: trastogi
# PWD : /Users/trastogi/ExtremeSSD/neoantigen-target-discovery
# Host: Darwin mac-pg60x57671.local 25.4.0 arm64
# -a       HLA-A02:01           Allele for prediction (multiple alleles with ,)
# -p       1                    Use pe


## Step 4: Parse and Rank Results

We parse the netMHCpan output and merge binding predictions back with 
our mutation annotations. We then rank candidates by %Rank_EL (eluted 
ligand rank), which reflects natural presentation on the cell surface 
and is the recommended ranking metric for neoantigen prioritization.

In [7]:
def parse_netmhcpan_output(filepath):
    """
    Parse netMHCpan output into a dataframe.
    Extracts Pos, Peptide, Score_EL, Rank_EL, and BindLevel.
    """
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or line.startswith("-") or line.startswith("Pos"):
                continue
            parts = line.split()
            if len(parts) < 12:
                continue
            try:
                pos = int(parts[0])
                peptide = parts[2]
                score_el = float(parts[11])
                rank_el = float(parts[12])
                bind_level = parts[13] if len(parts) > 13 else ""
                rows.append({
                    "pos": pos,
                    "peptide": peptide,
                    "score_el": score_el,
                    "rank_el": rank_el,
                    "bind_level": bind_level.replace("<=", "").strip()
                })
            except (ValueError, IndexError):
                continue
    return pd.DataFrame(rows)

results = parse_netmhcpan_output(netmhcpan_output)
print(f"Parsed {len(results)} predictions")
results.head()

Parsed 50973 predictions


,pos,peptide,score_el,rank_el,bind_level
0,1,AAAALAAAA,0.005655,8.862,
1,2,AAAASAAAA,0.002870,11.737,
2,3,AAAAAAAAA,0.003574,10.755,
3,4,AAAALAAAA,0.005655,8.862,
4,5,AAAAVAAAA,0.011577,6.476,


In [8]:
# Merge predictions with mutation annotations
peptides_df["pos"] = range(1, len(peptides_df) + 1)
merged = peptides_df.merge(results, on=["pos", "peptide"])

# Sort by rank_el ascending (lower = better binder)
merged = merged.sort_values("rank_el").reset_index(drop=True)

# Label binders
def label_binder(rank):
    if rank <= 0.5:
        return "Strong Binder"
    elif rank <= 2.0:
        return "Weak Binder"
    else:
        return "Non-binder"

merged["binding_call"] = merged["rank_el"].apply(label_binder)

# Summary
print("Binding summary:")
print(merged["binding_call"].value_counts())
print(f"\nTop 10 candidates:")
merged[["gene", "sample", "hgvsp", "peptide", "rank_el", "binding_call"]].head(10)

Binding summary:
binding_call
Non-binder    50973
Name: count, dtype: int64

Top 10 candidates:


,gene,sample,hgvsp,peptide,rank_el,binding_call
0,ZBTB46,MMRF_1720_1_BM_CD138pos_T1_KBS5U_L04823,p.V78I,AAAAIAAAA,5.557,Non-binder
1,FRAS1,MMRF_2683_1_BM_CD138pos_T1_KHS5U_L14407,p.V3453I,AAAAIAAAA,5.557,Non-binder
2,VSIG2,MMRF_1913_3_BM_CD138pos_T1_KHS5U_L16547,p.M193I,AAAAIAAAA,5.557,Non-binder
3,PGR,MMRF_1775_1_BM_CD138pos_T1_KBS5U_K11331,p.V611I,AAAAIAAAA,5.557,Non-binder
4,IGHV2-70,MMRF_1775_1_BM_CD138pos_T1_KBS5U_K11331,p.T38I,AAAAIAAAA,5.557,Non-binder
5,SHISA9,MMRF_1775_1_BM_CD138pos_T1_KBS5U_K11331,p.V215I,AAAAIAAAA,5.557,Non-binder
6,SEZ6L2,MMRF_1775_1_BM_CD138pos_T1_KBS5U_K11331,p.V370I,AAAAIAAAA,5.557,Non-binder
7,SPEF2,MMRF_2728_1_BM_CD138pos_T1_KHS5U_L14867,p.K1822I,AAAAIAAAA,5.557,Non-binder
8,CHP2,MMRF_1913_3_BM_CD138pos_T1_KHS5U_L16547,p.V186I,AAAAIAAAA,5.557,Non-binder
9,RFTN2,MMRF_1997_1_BM_CD138pos_T1_KHS5U_L08124,p.K273I,AAAAIAAAA,5.557,Non-binder


## Results and Limitations

No strong binders were identified in this proof-of-concept run. This is 
expected given the synthetic peptide construction approach used here.

**Why no binders?**

HLA-A*02:01 has strict anchor residue preferences:
- Position 2: Leucine or Methionine strongly preferred
- Position 9 (C-terminus): Valine or Leucine strongly preferred

Our synthetic peptides use Alanine as flanking context (e.g. `AAAALAAAA`), 
which does not satisfy these anchor requirements. The binding prediction 
is therefore driven by the flanks, not the mutant residue.

**What a production pipeline would do differently:**

1. Fetch the full protein sequence from UniProt or Ensembl using the gene 
   name and mutation position
2. Extract the true 8-11 residue window centered on the mutation
3. Run predictions across multiple HLA alleles per patient using real 
   HLA typing data
4. Integrate RNA expression to prioritize mutations in expressed genes

This notebook demonstrates the full pipeline architecture. The synthetic 
peptide step is the only placeholder.

## Next Steps

This pipeline is designed to be modular. Each step can be upgraded 
independently:

**Immediate improvements:**
- Replace synthetic peptides with true protein context via Ensembl REST API
- Add HLA-A, HLA-B, HLA-C predictions across multiple alleles
- Integrate RNA-seq expression data to filter for expressed mutations only

**Longer term:**
- Patient-specific HLA typing from WXS data
- Frameshift neoantigen prediction (separate extraction logic)
- TCR binding prediction for shortlisted candidates
- Integration with clinical response data from CoMMpass study

**Relevance to pre-Myeloma vaccine design:**
Somatic mutations in MGUS (precursor to Multiple Myeloma) are detectable 
years before disease onset. A pipeline like this, applied to MGUS mutation 
data with matched HLA typing, could identify peptide targets for a 
preventive vaccine before the cancer fully develops.

## Recurrent Mutations Across Patients

Vaccine targets are most valuable when they appear in multiple patients.
A recurrent mutation found in several patients suggests it may be 
clonally selected and functionally relevant to tumor growth.

We identify genes mutated in more than one patient as priority candidates.

In [12]:
# Find recurrent mutations across patients
recurrent = (
    missense.groupby(["Hugo_Symbol", "HGVSp_Short"])
    .agg(
        n_patients=("Tumor_Sample_Barcode", "nunique"),
        samples=("Tumor_Sample_Barcode", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values("n_patients", ascending=False)
)

# Filter for mutations in more than one patient
recurrent_multi = recurrent[recurrent["n_patients"] > 1]

print(f"Total unique missense mutations: {len(recurrent)}")
print(f"Mutations in more than one patient: {len(recurrent_multi)}")
print(f"\nTop recurrent mutations:")
recurrent_multi.head(20)

Total unique missense mutations: 45480
Mutations in more than one patient: 3687

Top recurrent mutations:


,Hugo_Symbol,HGVSp_Short,n_patients,samples
27104,NRAS,p.Q61R,74,"MMRF_1054_1_BM_CD138pos_T1_KBS5U_L03479, MMRF_..."
21491,KRAS,p.Q61H,73,"MMRF_1032_1_BM_CD138pos_T2_TSE61_L00053, MMRF_..."
27101,NRAS,p.Q61K,57,"MMRF_1037_1_BM_CD138pos_T2_TSE61_K02458, MMRF_..."
4236,BRAF,p.V640E,34,"MMRF_1020_3_BM_CD138pos_T1_KBS5U_L06452, MMRF_..."
21482,KRAS,p.G13D,33,"MMRF_1038_1_BM_CD138pos_T2_TSE61_K02459, MMRF_..."
21477,KRAS,p.G12D,31,"MMRF_1082_1_BM_CD138pos_T2_TSE61_K02709, MMRF_..."
27097,NRAS,p.G13R,24,"MMRF_1073_1_BM_CD138pos_T1_KHS5U_L08711, MMRF_..."
27100,NRAS,p.Q61H,17,"MMRF_1195_1_BM_CD138pos_T2_TSE61_K03425, MMRF_..."
21480,KRAS,p.G12V,15,"MMRF_1252_1_BM_CD138pos_T2_TSE61_L00074, MMRF_..."
21478,KRAS,p.G12R,13,"MMRF_1090_1_BM_CD138pos_T2_KAS5U_L02452, MMRF_..."


In [13]:
# Remove immunoglobulin genes (somatic hypermutation artifacts)
# Plasma cells undergo somatic hypermutation in IG genes as normal B-cell biology.
# These are not tumor drivers and should not be considered vaccine targets.
ig_prefixes = ("IGH", "IGL", "IGK", "IGLL", "IGLV", "IGHV", "IGHJ", "IGKV")
ig_mask = missense["Hugo_Symbol"].str.startswith(ig_prefixes)

print(f"IG gene variants removed: {ig_mask.sum()}")
print(f"Genes removed: {missense[ig_mask]['Hugo_Symbol'].unique().tolist()}")

missense_filtered = missense[~ig_mask].copy()
print(f"\nRemaining missense variants: {len(missense_filtered)}")

IG gene variants removed: 2935
Genes removed: ['IGLV3-1', 'IGHV2-70', 'IGLV3-25', 'IGLV3-21', 'IGKV4-1', 'IGKV3-20', 'IGLV4-60', 'IGLV7-46', 'IGLV5-45', 'IGHV2-70D', 'IGKJ4', 'IGHJ4', 'IGHJ3', 'IGHV1-69D', 'IGLV3-10', 'IGKV2-24', 'IGHJ5', 'IGHD7-27', 'IGHV3-21', 'IGHV4-34', 'IGLV6-57', 'IGKV1-8', 'IGLV4-3', 'IGLC3', 'IGLL5', 'IGKV5-2', 'IGKJ5', 'IGLV1-51', 'IGLV5-37', 'IGLV3-19', 'IGHV2-5', 'IGKV2-28', 'IGHV1-3', 'IGHV3-13', 'IGKV1-5', 'IGKV1D-8', 'IGHG1', 'IGHV3-30', 'IGLV4-69', 'IGLV1-47', 'IGHV3-7', 'IGLV3-16', 'IGHV1-69', 'IGHV1-2', 'IGHJ1', 'IGKV1-6', 'IGHV1-24', 'IGHV3-64', 'IGLV2-23', 'IGHD3-10', 'IGHJ6', 'IGKV1D-43', 'IGLV2-14', 'IGKJ2', 'IGKV2D-29', 'IGHV1-18', 'IGLV8-61', 'IGHV3-33', 'IGHD3-3', 'IGKV1-33', 'IGLV2-8', 'IGHD2-8', 'IGLC2', 'IGHV4-39', 'IGLV3-22', 'IGHA1', 'IGHD', 'IGLV3-9', 'IGLV3-12', 'IGHV3-23', 'IGHV5-51', 'IGHD3-9', 'IGKC', 'IGHD3-16', 'IGKV1D-13', 'IGHV3-16', 'IGHD5-24', 'IGHD6-13', 'IGHV1-45', 'IGHA2', 'IGHD3-22', 'IGLON5', 'IGHV3-53', 'IGKJ3', 'IGHV3-66',

In [14]:
# Rerun recurrence analysis on filtered data
recurrent_filtered = (
    missense_filtered.groupby(["Hugo_Symbol", "HGVSp_Short"])
    .agg(
        n_patients=("Tumor_Sample_Barcode", "nunique"),
        samples=("Tumor_Sample_Barcode", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values("n_patients", ascending=False)
)

recurrent_filtered_multi = recurrent_filtered[recurrent_filtered["n_patients"] > 1]

print(f"Total unique missense mutations (filtered): {len(recurrent_filtered)}")
print(f"Recurrent in >1 patient (filtered): {len(recurrent_filtered_multi)}")
print(f"\nTop 30 recurrent mutations (IG genes excluded):")
recurrent_filtered_multi[["Hugo_Symbol", "HGVSp_Short", "n_patients"]].head(30)

Total unique missense mutations (filtered): 43573
Recurrent in >1 patient (filtered): 3157

Top 30 recurrent mutations (IG genes excluded):


,Hugo_Symbol,HGVSp_Short,n_patients
25197,NRAS,p.Q61R,74
19584,KRAS,p.Q61H,73
25194,NRAS,p.Q61K,57
4236,BRAF,p.V640E,34
19575,KRAS,p.G13D,33
19570,KRAS,p.G12D,31
25190,NRAS,p.G13R,24
25193,NRAS,p.Q61H,17
19573,KRAS,p.G12V,15
19571,KRAS,p.G12R,13


## Biological Interpretation of Recurrent Mutations

After removing immunoglobulin gene variants (somatic hypermutation artifacts),
3,157 mutations are recurrent across the 1,091-patient cohort.

### RAS/MAPK Pathway Dominance

The top recurrent mutations cluster in a single oncogenic pathway:

- **NRAS p.Q61R** (74 patients), **p.Q61K** (57 patients), **p.Q61H** (17 patients)
- **KRAS p.Q61H** (73 patients), **p.G13D** (33 patients), **p.G12D** (31 patients)
- **BRAF p.V640E** (34 patients)

NRAS, KRAS, and BRAF are sequential nodes in the RAS/MAPK signaling cascade.
Activating mutations in any of these genes constitutively drive cell 
proliferation. Their dominance here is consistent with published myeloma 
genomics literature, validating the pipeline output.

Q61 mutations in RAS proteins are particularly significant: glutamine at 
position 61 is required for GTP hydrolysis. Mutations here lock RAS in its 
active GTP-bound state permanently, continuously signaling downstream.

For vaccine design, RAS/MAPK hotspot mutations are high-priority targets 
because they are clonal (present in all tumor cells), functionally required 
for tumor survival, and shared across many patients.

Note: BRAF p.V640E is the GRCh38 annotation of the canonical V600E mutation 
well-characterized in melanoma and other cancers.

### Myeloma-Specific Drivers

**IRF4 p.K123R** (12 patients) is a gain-of-function hotspot mutation specific 
to myeloma. IRF4 is a transcription factor essential for plasma cell survival. 
Myeloma cells are often dependent on IRF4 activity (oncogene addiction), making 
this a particularly relevant target.

**DIS3 p.D488N** (11 patients) and **p.R780K** (8 patients) are mutations in 
an RNA exonuclease that is among the most frequently mutated genes in myeloma 
specifically. Its myeloma enrichment makes it a disease-specific target.

### PIM1 p.K24N

PIM1 is a serine/threonine kinase that promotes cell survival and is 
overexpressed in myeloma. Recurrent mutations here suggest selective pressure 
to alter its activity in the tumor context.

### Caution: RPL10 p.I70L

RPL10 is a ribosomal protein expressed ubiquitously. While recurrent mutations 
have been reported in hematologic malignancies, the wildtype protein is present 
in all normal cells. Any vaccine targeting this mutation would require careful 
evaluation of the mutant vs wildtype peptide similarity to avoid autoimmunity.

### Priority Candidates for Next Steps

Based on recurrence, myeloma-specificity, and known biological relevance:

1. NRAS Q61 hotspot mutations
2. KRAS G12/G13/Q61 hotspot mutations  
3. BRAF V640E
4. IRF4 K123R
5. DIS3 D488N / R780K

These would be the first targets to run through a production pipeline with 
true protein sequence context and patient-specific HLA typing.